#  RAG-DataAfriqueHub - Quick Start Guide

Welcome! This notebook shows you how to use the RAG pipeline with a **simple local setup** using Ollama.

## Prerequisites

1. **Python 3.12+** installed
2. **Ollama** running locally (download from https://ollama.ai)
   ```bash
   ollama pull llama2
   ollama serve  # Keep this running in another terminal
   ```
3. **RAG-DataAfriqueHub** installed
   ```bash
   cd backend
   pip install -e .
   ```

## What you'll learn

✅ Load a RAG configuration  
✅ Ingest sample documents  
✅ Query the RAG pipeline  
✅ Inspect the retrieved sources  
✅ Customize the setup

## Step 1: Setup & Imports

In [ ]:
import sys
import yaml
from pathlib import Path

# Add backend to path
backend_path = Path("../").resolve()
sys.path.insert(0, str(backend_path))

# Import RAG components
from src.core.factory import RAGPipelineFactory
from src.core.models import Document, Query

print(" Imports successful!")
print(f"Working directory: {backend_path}")

## Step 2: Load Configuration

We'll use the **free.yaml** config which uses:
- **Ollama** for LLM (local, free)
- **BM25** for retrieval (lightweight)
- **Simple Vector Store** (in-memory)

In [ ]:
# Load configuration
config_path = "configs/free.yaml"

try:
    with open(config_path) as f:
        config = yaml.safe_load(f)
    print(" Configuration loaded:")
    print(yaml.dump(config, default_flow_style=False)[:500] + "...")
except FileNotFoundError:
    print(f" Config not found at {config_path}")
    print("Make sure you're running this from the backend/ directory")

## Step 3: Build the RAG Pipeline

The factory will create all components based on the config.

In [ ]:
# Create factory and build pipeline
try:
    factory = RAGPipelineFactory.load_config(config_path)
    pipeline = factory.build()
    print(" Pipeline built successfully!")
    print(f"\nPipeline configuration:")
    print(f"  - Loader: {pipeline.loader.__class__.__name__}")
    print(f"  - Chunker: {pipeline.chunker.__class__.__name__}")
    print(f"  - Embedder: {pipeline.embedder.__class__.__name__}")
    print(f"  - VectorStore: {pipeline.vector_store.__class__.__name__}")
    print(f"  - Retriever: {pipeline.retriever.__class__.__name__}")
    print(f"  - LLM: {pipeline.llm.__class__.__name__}")
except Exception as e:
    print(f" Error building pipeline: {e}")
    print("Making sure Ollama is running? Try: ollama serve")

## Step 4: Ingest Documents

Let's ingest some sample documents about Africa.

In [ ]:
# Create sample documents
sample_documents = {
    "africa_economy.txt": """Africa is home to over 1.4 billion people and is the world's second-largest continent. 
    The African economy is diverse, with major sectors including agriculture, mining, and technology. 
    Tech hubs in Lagos, Nairobi, and Cairo are driving innovation across the continent.
    The African Development Bank estimates the continent's GDP will exceed $2.1 trillion by 2025.""",
    
    "west_africa_opportunities.txt": """West Africa presents significant economic opportunities in digital transformation, 
    renewable energy, and agriculture technology. Countries like Nigeria, Ghana, and Côte d'Ivoire are leading 
    software development and fintech innovation. The region has a young population that is increasingly tech-savvy.
    Mobile money services like M-Pesa have revolutionized financial inclusion across the region.""",
    
    "education_africa.txt": """Education in Africa is transforming with the rise of digital learning platforms. 
    Organizations like Coursera and edX are expanding access to quality education. 
    African universities are increasingly focusing on STEM education to prepare students for the job market.
    The UNESCO estimates that Africa needs to add 6 million teachers by 2030."""
}

# Save sample documents to data/documents/
docs_dir = Path("data/documents")
docs_dir.mkdir(parents=True, exist_ok=True)

for filename, content in sample_documents.items():
    filepath = docs_dir / filename
    filepath.write_text(content)
    print(f" Created {filename}")

print(f"\n Sample documents saved to data/documents/")

In [ ]:
# Ingest documents
files_to_ingest = [
    "data/documents/africa_economy.txt",
    "data/documents/west_africa_opportunities.txt",
    "data/documents/education_africa.txt"
]

print(" Ingesting documents...\n")
for file_path in files_to_ingest:
    try:
        result = pipeline.ingest(file_path)
        print(f" {Path(file_path).name}: {result['chunks_count']} chunks created")
    except Exception as e:
        print(f" Error ingesting {file_path}: {e}")

print(f"\n Total documents in index: {pipeline.vector_store.count()}")

## Step 5: Query the Pipeline

Now let's ask some questions about African economy and opportunities.

In [ ]:
# Simple query
question = "What are the main economic opportunities in West Africa?"

print(f" Question: {question}\n")
print(" Searching for answer...\n")

try:
    response = pipeline.query(question, top_k=3)
    
    print(" Answer:")
    print("-" * 80)
    print(response['answer'])
    print("-" * 80)
    
    print(f"\n Sources ({len(response['sources'])} documents):")
    for i, source in enumerate(response['sources'], 1):
        print(f"\n[Source {i}]")
        print(f"Content: {source['content'][:150]}...")
        if 'metadata' in source:
            print(f"Source: {source['metadata'].get('source', 'N/A')}")
            
except Exception as e:
    print(f" Error querying: {e}")

In [ ]:
# Try multiple questions
questions = [
    "What is the estimated GDP of Africa by 2025?",
    "How is education transforming in Africa?",
    "Which countries are leading tech innovation in Africa?"
]

print(" Running multiple queries...\n")

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"Query {i}: {question}")
    print(f"{'='*80}")
    
    try:
        response = pipeline.query(question, top_k=2)
        print(f"\nAnswer:\n{response['answer']}")
        print(f"\nConfidence: Sources retrieved: {len(response['sources'])}")
    except Exception as e:
        print(f"Error: {e}")

## Step 6: Inspect Pipeline Internals

In [ ]:
# Inspect the retrieval process
query_text = "African tech innovation"

print(f"Query: {query_text}\n")

# Direct retrieval without LLM (to see what documents are found)
try:
    retrieved = pipeline.retriever.retrieve(query_text, top_k=3)
    print(f"Retrieved {len(retrieved)} documents:\n")
    
    for i, doc in enumerate(retrieved, 1):
        print(f"[Result {i}] Score: {doc.get('score', 'N/A')}")
        print(f"Content: {doc['content'][:200]}...")
        print()
except Exception as e:
    print(f"Error: {e}")

## Step 7: Customization Examples

In [ ]:
# Example: Query with different parameters
question = "What industries are growing in Africa?"

# Retrieve with different top_k values
for top_k in [1, 3, 5]:
    try:
        response = pipeline.query(question, top_k=top_k, temperature=0.5)
        print(f"\n🔍 Top-K={top_k}:")
        print(f"Sources considered: {len(response['sources'])}")
        print(f"Answer preview: {response['answer'][:100]}...")
    except Exception as e:
        print(f"Error with top_k={top_k}: {e}")

In [ ]:
# Example: Batch ingest more documents
new_doc_content = """Renewable energy in Africa is a rapidly growing sector. 
Countries like Morocco and South Africa are investing heavily in solar and wind energy. 
The continent has enormous solar potential, receiving more than 5 hours of sunlight per day on average.
This presents opportunities for clean energy solutions and job creation."""

new_doc_path = "data/documents/renewable_energy_africa.txt"
Path(new_doc_path).write_text(new_doc_content)

print(f"Ingesting new document: {new_doc_path}")
result = pipeline.ingest(new_doc_path)
print(f" Added {result['chunks_count']} chunks")
print(f"Total documents now: {pipeline.vector_store.count()}")

## Next Steps

 **You've successfully:**
- Loaded the RAG pipeline
- Ingested documents
- Queried the system
- Inspected retrieval results

### Try these next:

1. **Use a different config** (premium.yaml with OpenAI)
   ```python
   pipeline = RAGPipelineFactory.load_config("configs/premium.yaml").build()
   ```

2. **Add your own documents**
   ```python
   pipeline.ingest("your_document.txt")
   ```

3. **Fine-tune retrieval parameters**
   - Change `top_k` to get more/fewer sources
   - Adjust `temperature` for more/less creative answers

4. **Deploy as an API**
   ```bash
   python -m uvicorn src.api.main:app --reload
   ```

### Resources
- [README Production](../../README_PRODUCTION.md)
- [Architecture](../../ARCHITECTURE.md)
- [API Docs](../README_API.md)
- [Contributing](../../CONTRIBUTING.md)